# 05 · The held-out run

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/egumasa/lda2-final-template/blob/main/notebooks/05_test.ipynb)

Open the test set once, score what you committed to, and stop.

```
  01_build_pool_<track>  →  02_sample  →  02b_add_samples  →  03_annotate  →  04_develop  →▶ 05_test  →  06_report
```

| | |
|---|---|
| **Reads** | `data/gold/<track>_<group>_test.json` (from 03) · your prompt files · `..._rounds.json` (from 04) |
| **Writes** | `outputs/<track>_<group>_predictions.json` · `..._test_log.jsonl` · `..._rounds.json` (with the test row added) |

---

**This is the first time all week that `TEST_PATH` gets opened.** Your prompt is settled, you run it against items it has never been tuned against, and whatever comes out is what you report.

Expect it to be lower than your best dev round. That is the ordinary outcome, not a failure: the gap is roughly how much of your improvement was tuning to those particular dev items rather than to the task. Reporting the gap is a stronger finding than reporting a high number, and `06_report.ipynb` asks you for it.

A hosted model is only *best-effort* reproducible even at `temperature=0`, so each run is frozen to a file and every number in the report comes out of that file rather than out of this session's memory.

**Nothing here stops you running it again.** Stopping you would be the wrong design — a genuine mistake at four o'clock on the last day needs a way forward. So instead nothing is overwritten, and every scoring appends a line to a log that travels in your submission. A second attempt is allowed. It is just not invisible, and §5 of your report has to account for it.

## Setup — run this first

This cell mounts your Google Drive and finds your group's shared folder, `lda2-final-template`. Everything the project produces — the pool, the gold set, your prompts, the outputs — is an ordinary file in there, which is what makes it survive the runtime resetting *and* lets the rest of your group see it.

**One member sets the folder up once:**

1. That member runs the `git clone` line this cell prints if the folder is missing, which puts it in their own Drive.
2. They share it with the group (right-click ▸ *Share*), with edit access.
3. Everyone else opens *Shared with me*, right-clicks the folder, and chooses **Add shortcut to Drive** ▸ *My Drive*.

Keep that shortcut's name exactly `lda2-final-template`. It is what makes the same path work for all of you — if Drive renames it to `lda2-final-template (1)`, this cell will not find it.

From then on, open notebooks from the folder itself (*File ▸ Open notebook ▸ Drive*) rather than from the GitHub badge, so you are working on your group's copy and not a fresh one.

**Looking inside a helper.** The functions this cell imports are defined in `scripts/`. Two ways to read one, both the same ones you used on Day 2:

- `help(save_json)` prints its first line — what to pass in and what comes back — and the description of each argument. Typing `save_json(` and pressing **Shift+Tab** shows the same thing in a pop-up.
- To read the code itself, open `scripts/pipeline.py` from the **Files** panel on the left. Colab lists the functions in that file down the side, so you can click straight to the one you want.

In [ ]:
# ------------------------------------------------------------------
# SETUP — run me first. You are not expected to read it.
# ------------------------------------------------------------------
# This cell is plumbing, and it is the only cell in the project that is.
# It finds your group's shared folder in Google Drive, because everything
# this project keeps goes in there: a Colab runtime is wiped when it resets,
# and nobody else in your group can see inside it. Then it makes the
# project's own code importable. Run it and move on; nothing below asks you
# to have understood it.

FOLDER = "lda2-final-template"     # the shared folder, in every member's Drive

import os, sys

PROJECT = ".."                              # running locally: it is just above us

try:
    from google.colab import drive           # only exists inside Colab
except ImportError:
    pass
else:
    drive.mount("/content/drive")
    PROJECT = "/content/drive/MyDrive/" + FOLDER
    if not os.path.isdir(PROJECT):
        raise RuntimeError(
            "Could not find " + PROJECT + "\n\n"
            "Setting the folder up for your group? Run this in a new cell:\n"
            "  !git clone https://github.com/egumasa/lda2-final-template.git "
            + PROJECT + "\n"
            "then share the folder with the rest of your group.\n\n"
            "Someone else already did? Open Drive, find the folder under "
            "'Shared with me', right-click it, and choose 'Add shortcut to "
            "Drive'. Keep the name exactly " + FOLDER + ".")
    # Work inside the project folder, where the notebooks live.
    os.makedirs(PROJECT + "/notebooks", exist_ok=True)
    os.chdir(PROJECT + "/notebooks")

# scripts/ and config.py, by their real paths - so they are found from wherever
# this notebook happens to be working.
sys.path.append(PROJECT)
sys.path.append(PROJECT + "/scripts")

# Re-read config.yaml every time this cell runs. Without the reload, Python
# hands back the settings it read the FIRST time, and editing config.yaml
# would appear to do nothing until you restarted the runtime.
import importlib
import config
importlib.reload(config)

# Named one by one rather than with `import *`, so that every name a cell
# below uses can be traced back to the file it came from — config.yaml for
# these, scripts/ for the rest.
from config import (TRACK, GROUP, RUN, SEED, N_PER_CLASS, DEV, CODERS,
                    MEMBERS, LABELS_ORDER, TEMPERATURE, MODEL, ROOT, OUT_DIR,
                    POOL_PATH, DEMO_POOL_PATH, SAMPLE_PATH, GOLD_PATH,
                    SAMPLE_BEFORE_TOPUP_PATH, DEV_PATH, TEST_PATH,
                    DISAGREED_PATH, PRED_PATH, ROUNDS_PATH, NOTES_PATH,
                    TESTLOG_PATH, PROMPT_FILE, SHEET_PATH, TRIAGE_PATH,
                    describe)

# Files in, files out, and the connection to the model: all plumbing.
from pipeline import (load_gold, load_prompt, load_json, save_json, setup,
                      freeze_test_run, read_test_log)

describe()                  # what this notebook is working on


## Connect to the model

The same settings as `04_develop.ipynb`, from the same `config.yaml`. If this line does not match the one you iterated under, the held-out score is not comparable with the dev rounds you are about to put it beside.

In [ ]:
setup(temperature=TEMPERATURE, seed=SEED, model=MODEL)

> **Everything above comes from `config.yaml`** — one small file at the top of the repo, which you edit once as a group, and the only file in the plumbing you touch. That is deliberate: the seed that drew your sample has to be the seed you report, and five copies of a number in five notebooks is five chances for them to disagree. Your settings are also the filenames — `track: cars50`, `group: kimura`, `run: v1` means this notebook reads and writes `cars50_kimura_v1_...`. If the line it just printed is not your track, your group and your seed, fix `config.yaml` and re-run this cell.

## Step 1 — Commit, before you open anything

Fill this in **first**, and do not change it after the next cell has run.

You may test more than one prompt on the held-out set. It is a legitimate thing to do and this notebook supports it — but it is a **different claim** from testing one, and the difference is entirely in whether you said so in advance. One prompt tested is an estimate of how that prompt does on unseen items. Three tested and the best one reported is the best of three tries, and the number is optimistic by an amount nobody can calculate afterwards.

So: name every candidate here, say how you will pick between them, and **report all of their numbers in §3 — not only the winner's.** `PLAN.md` §8 should already say what you are about to type.

Each candidate is a **file**, because that is the only thing that survives from the last notebook, and because a prompt you cannot produce on request is a result nobody can check.

The rounds table from `04_develop.ipynb` is loaded here too, so the held-out row lands at the bottom of the dev trail rather than in a table of its own.

In [ ]:
# ══ STEP 1 · Name your candidates and your rule ═══════════════════════════
# Loads the dev trail from 04, and lists the prompt files you are about to
# test — with the rule for picking between them, written before you look.
# Creates: CANDIDATES, WINNER_RULE, f1_by_round, NOTES

# ✏️ How many prompts you test, and how you pick the winner. Both go in your
#    report, whichever way the numbers fall.

# name -> prompt file. One entry is the ordinary case.
CANDIDATES = {
    "v1 few-shot": ROOT / "prompts" / (TRACK + ".txt"),
}

# If there is more than one above, how do you choose? Decide now.
WINNER_RULE = "…"    # e.g. "highest macro-F1; ties go to the simpler prompt"

f1_by_round = load_json(ROUNDS_PATH, what="rounds",
                        made_by="notebook 04_develop")
NOTES = load_json(NOTES_PATH, what="round notes",
                  made_by="notebook 04_develop")

print(len(CANDIDATES), "candidate(s):", list(CANDIDATES))
print("best dev round so far:", round(max(f1_by_round.values()), 3))


## Step 2 — Open the test set and run

The cell below opens `TEST_PATH` and runs every candidate against it, freezing each one as it goes. For each, `freeze_test_run`:

1. Runs the prompt over the test items.
2. **Saves before scoring anything.** It never overwrites — a second run lands in `..._predictions_attempt2.json` beside the first, and both stay.
3. Reads that file straight back off disk and scores *those* predictions, which checks that the file you will quote in your report is the file you think it is.
4. Appends one line to the log in your submission bundle: the score, the attempt number, and a fingerprint of the prompt that produced it.
5. Adds the score to `f1_by_round` and saves the table.

It also freezes the model's **raw replies** beside the predictions. The predictions are our reading of those replies; if a `??` turns out to matter, the reply is the evidence and it is gone as soon as this runtime resets.

**One person runs this.** It is the run you will be defending.

In [ ]:
# ══ STEP 2 · The held-out run ═════════════════════════════════════════════
# Opens the held-out items — the only cell all week that does — and freezes,
# scores and logs each candidate against them. One predictions file each.
# Creates: test, best_dev

test = load_gold(TEST_PATH)      # the first time this file is opened all week

# Your best DEV round, taken BEFORE any test row joins the table. Reading it
# inside the loop would compare the second candidate against the first
# candidate's held-out score, which is not a dev/test gap at all.
best_dev = max(f1_by_round.values())

# One note for every candidate: with more than one, the rule you committed
# to in step 1 is the thing the log has to carry.
note = ""
if len(CANDIDATES) > 1:
    note = WINNER_RULE

for name in CANDIDATES:
    print()
    print("=" * 70)
    print("candidate:", name)
    print("=" * 70)
    freeze_test_run(load_prompt(CANDIDATES[name]), test, f1_by_round,
                    PRED_PATH, TESTLOG_PATH, ROUNDS_PATH, PROMPT_FILE,
                    dev_f1=best_dev,
                    ordered=False,   # True only if your labels are a SCALE
                    labels=LABELS_ORDER,
                    key="TEST · " + name,
                    note=note)

    # The reason this candidate was tested at all, so report section 2 has
    # a line for the held-out row rather than a bare number at the bottom.
    NOTES["TEST · " + name] = "held out · " + WINNER_RULE

save_json(NOTES, NOTES_PATH, what="round notes", overwrite=True)


### The log, as it now stands

One line per scoring. **If there is more than one, report §5 has to account for every one of them** — which is the whole reason this file exists rather than a lock on the cell above.

Look at `prompt_sha1`. Two rows with the *same* fingerprint are one prompt run twice, which tells you something useful about how much the model varies on its own. Two rows with *different* fingerprints are two different prompts — fine if you named both in step 1, and a prompt tuned after seeing the held-out set if you did not.

In [ ]:
read_test_log(TESTLOG_PATH)

**✍️ For your report and the Q&A** — this goes in the write-up, not in a cell below.

> We tested ___ prompt(s) on the held-out set, decided in advance, and picked between them by ___.
>
> Their held-out scores were ___.
>
> Our best dev round was ___ and the held-out score was ___, a gap of ___, which we read as ___.

The gap is the finding, in either direction. A held-out score well below the dev trail says some of the gain was tuning to those particular dev items; one that matches says the change generalised. Both are reportable, and a report that quotes only the dev number is quoting the one you cannot defend.

If the log has more than one line, say here what the second run was and why.

---

**Next:** `06_report.ipynb`. It loads the files you just wrote and nothing else — so from here on, your numbers cannot move.